# OPD: Why Let the Student Write First, Then Let the Teacher Grade?

> **Background**: Frontier models (like GPT-4) are very capable, but they can be expensive and slow. We often want to "compress" that ability into a smaller model. A traditional approach is: let the big model write a dataset of perfect answers, then let the small model memorize them. But even if the student memorizes well, it still fails once it has to write on its own, because it never practiced recovering from its own mistakes.
>
> Goal for this part: **understand the core idea of On-Policy Distillation (OPD)**: the student writes first, and the teacher corrects it on-the-fly, instead of the student memorizing the teacher's finished answers.

One-sentence intuition:

**Offline distillation = the student memorizes essays written by the teacher; OPD = the student writes an essay, and the teacher grades it line by line.**


In [1]:
import numpy as np

np.random.seed(42)


### 1. Basics first: what is knowledge distillation?

You have a large model (the **Teacher**) and a small model (the **Student**).
The goal is to make the small model answer like the large model.

```text
Teacher (GPT-4, 1.8T params):  input "1+1=?" -> output "2"  [correct]
Student (small, 0.5B params):  input "1+1=?" -> output "3"  [wrong]

Distillation goal: make the Student also output "2".
```

Two high-level approaches:

- **Offline distillation**: the Teacher writes a big "answer book" first, then the Student memorizes it.
- **On-policy distillation (OPD)**: the Student answers first, then the Teacher grades the Student's actual trajectory.


### 2. The big picture: how are these training styles different?

You can distinguish the main methods by two questions:

1. During training, **who wrote the prefix** (the context tokens you condition on)?
2. **Who provides the target** you train toward?

| Method | Prefix source | Target source | Analogy |
|:---|:---|:---|:---|
| **SFT** | dataset gold prefix | dataset gold target | memorizing a textbook |
| **Offline distillation** | Teacher-written fixed prefix | Teacher probability distribution | memorizing the teacher's model answer |
| **OPD** | **Student-generated prefix** | Teacher distribution on the student's prefix | student writes, teacher grades |
| **OPSD** | **Student-generated prefix** | the same model with privileged info | closed-book student taught by open-book self |

The key difference is the prefix column: in OPD/OPSD, the prefix is produced by the Student itself.


### 3. The root problem: Exposure Bias

This is the most important concept for understanding why OPD is needed.

```text
During training (SFT / offline distillation):
  prefixes are always the first half of a correct answer
  e.g. "I yesterday went" -> learn to predict "school"
  the student only sees grammatical, coherent prefixes

During inference (student generates its own tokens):
  the student can drift and produce weird prefixes
  e.g. "I yesterday very park" -> now what token comes next?
  the student never saw prefixes like this during training
  -> it doesn't know how to recover -> collapses
```

That mismatch is **exposure bias**: training happens on "clean" prefixes, inference happens on the student's own (possibly messy) prefixes.

Like learning to drive only on a perfectly flat highway, then panicking the first time you hit a bumpy dirt road.


In [2]:
# Simulated vocabulary
vocab = {
    "I": 0,
    "yesterday": 1,
    "went": 2,
    "school": 3,
    "park": 4,
    "store": 5,
    "very": 6,
    "happy": 7,
    "bored": 8,
    "tired": 9,
}

id2token = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)


def softmax(logits):
    logits = np.array(logits, dtype=np.float64)
    exp_logits = np.exp(logits - logits.max())
    return exp_logits / exp_logits.sum()


def log_softmax(logits):
    logits = np.array(logits, dtype=np.float64)
    m = logits.max()
    return logits - m - np.log(np.exp(logits - m).sum())


def student_model(input_ids):
    """Toy student model: returns logits given a prefix."""
    rng = np.random.RandomState(sum(input_ids) * 7 + 42)
    return rng.randn(vocab_size) * 2


def teacher_model(input_ids):
    """Toy teacher model: returns logits given a prefix."""
    rng = np.random.RandomState(sum(input_ids) * 13 + 7)
    return rng.randn(vocab_size) * 1.2


print("Vocab + toy teacher/student models are ready. [ok]")


Vocab + toy teacher/student models are ready. [ok]


In [3]:
# Exposure bias demo: compare training prefixes vs generated prefixes
print("=== Exposure Bias demo ===")
print()

train_prefix = [0, 1, 2]  # "I yesterday went"
print(f"Training prefix: {[id2token[i] for i in train_prefix]}  (clean prefix from data)")

t_logits = teacher_model(train_prefix)
t_probs = softmax(t_logits)
top3 = np.argsort(t_probs)[-3:][::-1]
print("  Under this prefix, the teacher's top-3 next tokens are:")
for idx in top3:
    idx = int(idx)
    print(f"    '{id2token[idx]}': {t_probs[idx]:.3f}")
print()

print("Inference-time: the student generates autoregressively:")
np.random.seed(99)
student_sequence = [0]  # "I"
for step in range(3):
    s_logits = student_model(student_sequence)
    s_probs = softmax(s_logits)
    next_token = np.random.choice(vocab_size, p=s_probs)
    student_sequence.append(next_token)
    print(
        f"  Step {step+1}: input {[id2token[i] for i in student_sequence[:-1]]} "
        f"-> sampled '{id2token[next_token]}' (p={s_probs[next_token]:.3f})"
    )
print()

weird_prefix = student_sequence[:-1]
s_logits_weird = student_model(weird_prefix)
s_probs_weird = softmax(s_logits_weird)
entropy = -np.sum(s_probs_weird * np.log(s_probs_weird + 1e-10))

print(f"Student-generated prefix: {[id2token[i] for i in weird_prefix]}")
print(f"  Output entropy: {entropy:.3f} (higher = more uncertain)")
print(f"  Top token: '{id2token[int(np.argmax(s_probs_weird))]}' ({np.max(s_probs_weird):.3f})")
print()
print("-> The student never trained on prefixes like this. The next-token distribution becomes uncertain.")
print("-> Exposure bias: training prefix distribution != inference prefix distribution.")


=== Exposure Bias demo ===

Training prefix: ['I', 'yesterday', 'went']  (clean prefix from data)
  Under this prefix, the teacher's top-3 next tokens are:
    'yesterday': 0.257
    'tired': 0.164
    'went': 0.157

Inference-time: the student generates autoregressively:
  Step 1: input ['I'] -> sampled 'very' (p=0.386)
  Step 2: input ['I', 'very'] -> sampled 'went' (p=0.518)
  Step 3: input ['I', 'very', 'went'] -> sampled 'went' (p=0.190)

Student-generated prefix: ['I', 'very', 'went']
  Output entropy: 1.019 (higher = more uncertain)
  Top token: 'yesterday' (0.682)

-> The student never trained on prefixes like this. The next-token distribution becomes uncertain.
-> Exposure bias: training prefix distribution != inference prefix distribution.


### 4. OPD's solution: learn on your own trajectory

OPD has three core steps:

```text
Step 1: Student generates an answer (rollout)
  Prompt: "I yesterday" -> Student: "I yesterday went park very happy"

Step 2: Teacher scores each next-token decision under the Student's prefix
  position 2: prefix=[I, yesterday], Student wrote "went" -> Teacher: ok
  position 3: prefix=[I, yesterday, went], Student wrote "park" -> Teacher: good
  position 4: prefix=[I, yesterday, went, park], Student wrote "very" -> Teacher: not great

Step 3: Update the Student using Teacher feedback
  tokens the Teacher likes -> increase probability
  tokens the Teacher dislikes -> decrease probability
```

Key point: the Student gets corrected on **the prefixes it actually produced**.
Even if it produces a weird prefix, the Teacher still provides guidance on that exact prefix.


In [4]:
# OPD core flow: student rollout + teacher grading on student prefixes
print("=== OPD core flow demo ===")
print()

prompt = [0, 1]  # "I yesterday"
sequence = prompt.copy()
temperature = 1.0

print(f"Prompt: {[id2token[i] for i in prompt]}")
print()

print("Step 1: Student rollout")
for _ in range(3):
    logits = student_model(sequence)
    probs = softmax(logits / temperature)
    next_token = np.random.choice(vocab_size, p=probs)
    sequence.append(next_token)
    print(f"  prefix={[id2token[i] for i in sequence[:-1]]} -> sampled {id2token[next_token]}")

print()
print(f"Generated sequence: {[id2token[i] for i in sequence]}")
print()

print("Step 2: Teacher evaluates on the student's prefixes")
for t in range(len(prompt), len(sequence)):
    prefix = sequence[:t]
    sampled_token = sequence[t]

    s_logits = student_model(prefix)
    t_logits = teacher_model(prefix)
    s_logp = log_softmax(s_logits)[sampled_token]
    t_logp = log_softmax(t_logits)[sampled_token]

    advantage = t_logp - s_logp

    print(f"  position {t}: prefix={[id2token[i] for i in prefix]}")
    print(f"    student wrote '{id2token[sampled_token]}': s_logp={s_logp:.3f}, t_logp={t_logp:.3f}")
    print(f"    advantage = {advantage:+.3f} -> {'reward' if advantage > 0 else 'penalize'}")

print()
print("Key point: no matter how weird the student's prefix is, the teacher gives feedback on that exact prefix.")


=== OPD core flow demo ===

Prompt: ['I', 'yesterday']

Step 1: Student rollout
  prefix=['I', 'yesterday'] -> sampled went
  prefix=['I', 'yesterday', 'went'] -> sampled very
  prefix=['I', 'yesterday', 'went', 'very'] -> sampled park

Generated sequence: ['I', 'yesterday', 'went', 'very', 'park']

Step 2: Teacher evaluates on the student's prefixes
  position 2: prefix=['I', 'yesterday']
    student wrote 'went': s_logp=-3.118, t_logp=-2.268
    advantage = +0.850 -> reward
  position 3: prefix=['I', 'yesterday', 'went']
    student wrote 'very': s_logp=-2.144, t_logp=-2.923
    advantage = -0.779 -> penalize
  position 4: prefix=['I', 'yesterday', 'went', 'very']
    student wrote 'park': s_logp=-0.636, t_logp=-5.154
    advantage = -4.518 -> penalize

Key point: no matter how weird the student's prefix is, the teacher gives feedback on that exact prefix.


### 5. The math view: Forward KL vs Reverse KL

A useful way to understand offline distillation vs OPD is the **direction** of the KL divergence.
KL measures a distance between two distributions, but it is **asymmetric**:

$$D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$$

**Forward KL (offline distillation)**:

$$D_{KL}(P_{Teacher} \| P_{Student})$$

- expectation under the Teacher: whatever the Teacher cares about, the Student must cover
- behavior: **mode-covering** (cover all Teacher modes)
- risk for small models: capacity is limited, so it learns a little bit of everything and becomes "averaged"

**Reverse KL (OPD)**:

$$D_{KL}(P_{Student} \| P_{Teacher})$$

- expectation under the Student: you only pay loss where the Student actually goes
- behavior: **mode-seeking** (find one good mode you can generate stably)
- advantage for small models: you can specialize instead of covering everything.


In [5]:
# Build intuition for Forward KL vs Reverse KL with a toy distribution
print("=== Forward KL vs Reverse KL (intuition) ===")
print()

teacher_probs = np.array([0.40, 0.35, 0.25])
styles = ["academic", "humorous", "concise"]

print("Teacher believes good answers come in three styles:")
for style, prob in zip(styles, teacher_probs):
    bar = "#" * int(prob * 40)
    print(f"  {style:8s}: {prob:.0%} {bar}")
print()

student_avg_probs = np.array([0.33, 0.33, 0.34])
forward_kl = np.sum(
    teacher_probs * (np.log(teacher_probs + 1e-10) - np.log(student_avg_probs + 1e-10))
)

print("Forward KL (offline distillation): student tries to cover all modes:")
for style, prob in zip(styles, student_avg_probs):
    bar = "#" * int(prob * 40)
    print(f"  {style:8s}: {prob:.0%} {bar}")
print(f"  Forward KL = {forward_kl:.4f}")
print("  -> output becomes a weird mix: 'According to research... haha... in short...' ")
print()

student_spec_probs = np.array([0.05, 0.85, 0.10])
reverse_kl = np.sum(
    student_spec_probs * (np.log(student_spec_probs + 1e-10) - np.log(teacher_probs + 1e-10))
)

print("Reverse KL (OPD): student specializes in one mode:")
for style, prob in zip(styles, student_spec_probs):
    bar = "#" * int(prob * 40)
    print(f"  {style:8s}: {prob:.0%} {bar}")
print(f"  Reverse KL = {reverse_kl:.4f}")
print("  -> stable, high-quality humorous answers")
print()

print(f"Compare: Forward KL ({forward_kl:.4f}) vs Reverse KL ({reverse_kl:.4f})")
print("OPD is often friendlier to small models: you do not need to cover everything, just be good at one reliable mode.")


=== Forward KL vs Reverse KL (intuition) ===

Teacher believes good answers come in three styles:
  academic: 40% ################
  humorous: 35% ##############
  concise : 25% ##########

Forward KL (offline distillation): student tries to cover all modes:
  academic: 33% #############
  humorous: 33% #############
  concise : 34% #############
  Forward KL = 0.0207
  -> output becomes a weird mix: 'According to research... haha... in short...' 

Reverse KL (OPD): student specializes in one mode:
  academic: 5% ##
  humorous: 85% ##################################
  concise : 10% ####
  Reverse KL = 0.5586
  -> stable, high-quality humorous answers

Compare: Forward KL (0.0207) vs Reverse KL (0.5586)
OPD is often friendlier to small models: you do not need to cover everything, just be good at one reliable mode.


### 6. OPSD: OPD without an external Teacher

OPD needs a strong Teacher model. OPSD (On-Policy Self-Distillation) does not:
**Teacher and Student are the same model, but the Teacher sees extra information.**

```text
OPSD setup:

Student (closed-book): only sees the question
  -> writes an answer

Teacher (open-book): same model, but also sees the gold answer / solution steps
  -> provides next-token supervision on the Student's prefix

Training goal: make the closed-book Student match the open-book Teacher
-> internalize what you only knew when you could see the answer
```

Intuition: when you read the solution, you think "oh, that's how it works". But you forget once you close the book.
OPSD is "open-book you" repeatedly teaching "closed-book you".

Limitation: if the extra info is never available at test time (e.g., per-problem hidden solutions), OPSD may only learn an average strategy.
So OPSD is better for internalizing **shared rules** (format preferences, system prompts) than **unique per-example information**.


### 7. Three granularities of Teacher feedback

When the Teacher provides token-level supervision, it can provide different amounts of information:

1. **sampled-token**: only the token the Student actually sampled
2. **top-k**: the Teacher's top-k candidates
3. **full-vocab**: the Teacher's full probability distribution

More information is usually better, but more information can be harder to get in practice:

- API-based Teachers often give only sampled tokens (no logits)
- White-box Teachers can give logits for top-k or full vocabulary

Next we'll build an intuition with a tiny toy vocabulary.


In [6]:
# Compare three teacher feedback granularities
prefix_demo = [0, 1, 2]  # "I yesterday went"

print("=== Three granularities of teacher signal ===")
print()

t_logits = teacher_model(prefix_demo)
t_logprobs = log_softmax(t_logits)
sampled_token = 3  # "school"

print("1) sampled-token:")
print(f"   Teacher only tells you the score for '{id2token[sampled_token]}': {t_logprobs[sampled_token]:.3f}")
print("   You learn nothing about the other tokens.")
print()

k = 5
topk_idx = np.argsort(t_logprobs)[-k:][::-1]
print(f"2) top-{k}:")
for idx in topk_idx:
    idx = int(idx)
    print(f"   Teacher likes '{id2token[idx]}' with logp {t_logprobs[idx]:.3f}")
print()

print("3) full-vocab:")
print(f"   Teacher provides probabilities for all {vocab_size} tokens")
all_probs = softmax(t_logits)
for i, tok in id2token.items():
    bar = "#" * int(all_probs[i] * 40)
    print(f"   {tok:8s}: {all_probs[i]:.4f}  {bar}")


=== Three granularities of teacher signal ===

1) sampled-token:
   Teacher only tells you the score for 'school': -3.794
   You learn nothing about the other tokens.

2) top-5:
   Teacher likes 'yesterday' with logp -1.357
   Teacher likes 'tired' with logp -1.806
   Teacher likes 'went' with logp -1.849
   Teacher likes 'I' with logp -2.133
   Teacher likes 'park' with logp -2.340

3) full-vocab:
   Teacher provides probabilities for all 10 tokens
   I       : 0.1185  ####
   yesterday: 0.2573  ##########
   went    : 0.1575  ######
   school  : 0.0225  
   park    : 0.0963  ###
   store   : 0.0475  #
   very    : 0.0538  ##
   happy   : 0.0298  #
   bored   : 0.0525  ##
   tired   : 0.1643  ######


### 8. If you only have sampled-token: KL estimators

In many realistic settings you cannot get full Teacher logits. You may only observe:

- what token the Student sampled
- (maybe) the Teacher's probability / logprob for that token

So we need estimators of KL-related losses from limited signals.
We'll compare three common estimators:

- **k1**: unbiased but can be negative (high variance)
- **k2**: biased but always non-negative (lower variance)
- **k3**: unbiased + non-negative + lower variance (often recommended)


In [7]:
def k1_estimator(s_logp, t_logp):
    """Direct difference: unbiased but high-variance, can be negative."""
    return s_logp - t_logp


def k2_estimator(s_logp, t_logp):
    """Quadratic approximation: biased but always non-negative, lower variance."""
    diff = s_logp - t_logp
    return 0.5 * diff * diff


def k3_estimator(s_logp, t_logp):
    """k3: unbiased + non-negative + lower variance (often recommended)."""
    ratio = t_logp - s_logp  # log(P_T / P_S)
    r = np.exp(ratio)        # P_T / P_S
    return r - ratio - 1


print("=== Three KL estimators (toy comparison) ===")
print(f"{'s_logp':>10} {'t_logp':>10} {'k1':>10} {'k2':>10} {'k3':>10}  note")
print("-" * 68)

cases = [
    (-0.5, -0.5, "match"),
    (-1.0, -0.5, "teacher prefers"),
    (-0.5, -1.0, "student prefers"),
    (-2.0, -0.5, "big gap (teacher prefers)"),
    (-0.5, -2.0, "big gap (student prefers)"),
    (-3.0, -0.1, "extreme gap"),
]

for s_lp, t_lp, desc in cases:
    k1 = k1_estimator(s_lp, t_lp)
    k2 = k2_estimator(s_lp, t_lp)
    k3 = k3_estimator(s_lp, t_lp)
    print(f"{s_lp:>+10.2f} {t_lp:>+10.2f} {k1:>+10.4f} {k2:>10.4f} {k3:>10.4f}  <- {desc}")

print()
print("k1 can be negative -> not ideal as a loss")
print("k2 is always >= 0  -> biased but stable")
print("k3 is always >= 0  -> unbiased and stable (recommended)")


=== Three KL estimators (toy comparison) ===
    s_logp     t_logp         k1         k2         k3  note
--------------------------------------------------------------------
     -0.50      -0.50    +0.0000     0.0000     0.0000  <- match
     -1.00      -0.50    -0.5000     0.1250     0.1487  <- teacher prefers
     -0.50      -1.00    +0.5000     0.1250     0.1065  <- student prefers
     -2.00      -0.50    -1.5000     1.1250     1.9817  <- big gap (teacher prefers)
     -0.50      -2.00    +1.5000     1.1250     0.7231  <- big gap (student prefers)
     -3.00      -0.10    -2.9000     4.2050    14.2741  <- extreme gap

k1 can be negative -> not ideal as a loss
k2 is always >= 0  -> biased but stable
k3 is always >= 0  -> unbiased and stable (recommended)


### 9. Full OPD training step (put it all together)

Now we will simulate one complete OPD step:

1. Student rolls out tokens on its own prefix
2. Teacher scores the Student's decisions under the Student prefix
3. We compute an OPD loss and (in real training) backprop to update the Student


In [8]:
def simulate_opd_training(prompt_ids, num_gen_tokens=3, mode="sampled_token_k3", topk=5):
    """Simulate one full OPD training step."""

    # Step 1: student rollout
    sequence = list(prompt_ids).copy()
    temp = 1.0

    print("Step 1: Rollout (student generates)")
    print(f"  Prompt: {[id2token[i] for i in prompt_ids]}")
    for _ in range(num_gen_tokens):
        logits = student_model(sequence)
        probs = softmax(logits / temp)
        next_token = np.random.choice(vocab_size, p=probs)
        sequence.append(next_token)

    generated = sequence[len(prompt_ids):]
    print(f"  Generated: {[id2token[i] for i in generated]}")

    print()
    print(f"Step 2: compute OPD loss (mode: {mode})")
    total_loss = 0.0

    for t in range(len(prompt_ids), len(sequence)):
        prefix = sequence[:t]
        sampled_token = sequence[t]

        if mode == "sampled_token_k3":
            s_logp = log_softmax(student_model(prefix))[sampled_token]
            t_logp = log_softmax(teacher_model(prefix))[sampled_token]
            loss = k3_estimator(s_logp, t_logp)
            print(f"  pos {t}: '{id2token[sampled_token]}' | s_logp={s_logp:.3f} t_logp={t_logp:.3f} | k3={loss:.4f}")

        elif mode == "topk_rkl":
            t_logp_full = log_softmax(teacher_model(prefix))
            topk_idx = np.argsort(t_logp_full)[-topk:][::-1]

            t_renorm = softmax(t_logp_full[topk_idx])
            s_logp_full = log_softmax(student_model(prefix))
            s_renorm = softmax(s_logp_full[topk_idx])

            loss = np.sum(
                s_renorm * (np.log(s_renorm + 1e-10) - np.log(t_renorm + 1e-10))
            )
            print(f"  pos {t}: top-{topk}={[id2token[int(i)] for i in topk_idx]} | RKL={loss:.4f}")

        else:
            raise ValueError(f"unknown mode: {mode}")

        total_loss += float(loss)

    avg_loss = total_loss / max(1, len(generated))
    print()
    print(f"Total loss: {total_loss:.4f} | avg: {avg_loss:.4f}")
    print("-> Backprop gradients, update the Student -> next rollout -> repeat")
    return avg_loss


np.random.seed(42)
print("### Mode: sampled_token + k3 ###")
simulate_opd_training([0, 1], num_gen_tokens=3, mode="sampled_token_k3")


### Mode: sampled_token + k3 ###
Step 1: Rollout (student generates)
  Prompt: ['I', 'yesterday']
  Generated: ['school', 'store', 'store']

Step 2: compute OPD loss (mode: sampled_token_k3)
  pos 2: 'school' | s_logp=-0.803 t_logp=-5.509 | k3=3.7149
  pos 3: 'store' | s_logp=-0.685 t_logp=-4.153 | k3=2.4994
  pos 4: 'store' | s_logp=-2.903 t_logp=-4.291 | k3=0.6372

Total loss: 6.8515 | avg: 2.2838
-> Backprop gradients, update the Student -> next rollout -> repeat


2.283848208244739

In [9]:
np.random.seed(42)
print("### Mode: top-k Reverse KL ###")
simulate_opd_training([0, 1], num_gen_tokens=3, mode="topk_rkl", topk=5)


### Mode: top-k Reverse KL ###
Step 1: Rollout (student generates)
  Prompt: ['I', 'yesterday']


  Generated: ['school', 'store', 'store']

Step 2: compute OPD loss (mode: topk_rkl)
  pos 2: top-5=['very', 'I', 'store', 'bored', 'tired'] | RKL=1.4217
  pos 3: top-5=['happy', 'park', 'yesterday', 'went', 'bored'] | RKL=0.5603
  pos 4: top-5=['school', 'happy', 'bored', 'I', 'yesterday'] | RKL=0.3147

Total loss: 2.2966 | avg: 0.7655
-> Backprop gradients, update the Student -> next rollout -> repeat


0.7655468551277211

### 10. Why OPD became popular recently: engineering infrastructure matured

OPD is conceptually simple, but production OPD needs:

- fast rollout generation (e.g., vLLM)
- efficient distributed training (e.g., DeepSpeed/FSDP)
- stable KL / distillation objectives
- tokenizer alignment across Teacher/Student (sometimes different vocabularies)

These pieces became solid enough in 2024-2026, so OPD moved from papers into mainstream post-training pipelines.


### 11. Quick paper tour (as of 2026-05)

A minimal reading list to orient yourself:

- **GKD** (Generative Knowledge Distillation)
- **OPSD** (On-Policy Self-Distillation)
- **Pitfalls** (when OPD goes wrong)
- **OGLS-SD** (improving stability / objectives)

The exact titles are less important than the recurring pattern:
SFT -> RL/GRPO -> OPD (integrate/compress).


### 12. Landscape: categorizing 21 core OPD papers (as of 2026-05)

A useful way to organize the literature is by what kind of Teacher signal you have and what your goal is.

Below is a compact taxonomy (not exhaustive):

1. Foundations: define OPD-style objectives, KL estimators, and training recipes.
2. Bridging the gap: reduce mismatch between rollout distribution and training distribution.
3. Stability: make on-policy training numerically stable and robust.
4. Self-distillation: OPSD-style, where the Teacher is the same model with privileged info.
5. Context internalization: absorb rules/instructions into weights.
6. Efficiency: reduce rollout/training cost (multi-teacher, top-k, batching tricks).

Takeaway: the field keeps rediscovering the same core tension:

- on-policy prefixes are realistic, but noisy
- dense feedback helps, but can be expensive
- stability is the real engineering battle


### 13. Two axes to understand the OPD ecosystem

Axis A: Teacher type

- white-box Teacher (logits available)
- black-box Teacher (API only)
- self-Teacher (open-book vs closed-book)
- multi-Teacher
- context-based Teacher (privileged context)

Axis B: Goal

- compression (big -> small)
- integration (merge multiple experts)
- continual learning (preserve skills while adding new ones)
- RL substitute or RL complement

Once you place a method on these two axes, the design choices become much easier to reason about.


### 14. Industrial adoption: who uses OPD? (2024-2026)

A common pattern in modern post-training is:

SFT -> RL/GRPO -> OPD (integration / compression)

OPD often appears in the last stage to:

- merge multiple expert models into one smaller general model
- compact a model after RL
- distill capabilities that are expensive to run at inference time


---

## OPD Summary (extended)

### Core ideas (Sections 1-9)

1. Knowledge distillation = big model teaches small model
2. Offline distillation = student memorizes teacher-written prefixes
3. Exposure bias = training prefixes differ from inference prefixes
4. OPD = student rollouts + teacher feedback on the student's prefixes
5. Forward KL (offline) = mode-covering -> averaged outputs for small models
6. Reverse KL (OPD) = mode-seeking -> specialization is OK
7. OPSD = self-distillation with privileged information
8. Three granularities: sampled-token < top-k < full-vocab
9. Three KL estimators: k1 (unbiased but can be negative), k2 (biased but >=0), k3 (unbiased and >=0, recommended)

### Engineering & ecosystem (Sections 10-15)

10. Infrastructure: rollout engines + distributed training + tokenizer alignment
11. Quick paper tour: GKD / OPSD / Pitfalls / OGLS-SD
12. Landscape: organize papers by teacher signal and goal
13. Taxonomy axes: Teacher type and goal
14. Industrial adoption: OPD as a standard late-stage component
15. Tooling: TRL (entry) / veRL (production) / NeMo-RL (tokenizer alignment)

### One-sentence takeaway

Offline distillation = memorize the teacher's finished answers.
OPD = write first, then get graded on your own trajectory.

### References

- awesome-on-policy-distillation: https://github.com/chrisliu298/awesome-on-policy-distillation
- OPD Survey: https://arxiv.org/abs/2604.00626
- MiniLLM: https://arxiv.org/abs/2306.08543
- GKD: https://arxiv.org/abs/2306.13649
- ExOPD: https://arxiv.org/abs/2602.12125
- OPSD: https://arxiv.org/abs/2601.18734
- Pitfalls (Zhu et al.): https://arxiv.org/abs/2605.11182
- OGLS-SD: https://arxiv.org/abs/2605.12400
- Thinking Machines blog: https://thinkingmachines.ai/blog/on-policy-distillation/
- TRL DistillationTrainer: https://huggingface.co/docs/trl


---

## OPD Summary (short)

1. Knowledge distillation = big model teaches small model
2. Offline distillation = memorize teacher-written prefixes
3. Exposure bias = training prefixes differ from inference prefixes
4. OPD = student rollouts + teacher feedback on student prefixes
5. Forward KL (offline) = mode-covering -> small model becomes averaged
6. Reverse KL (OPD) = mode-seeking -> small model can specialize
7. OPSD = self-distillation with privileged info
8. Granularity: sampled-token < top-k < full-vocab
9. KL estimators: k1 (unbiased, can be negative), k2 (biased, >=0), k3 (unbiased, >=0, recommended)
10. Engineering: rollout engines + distributed training make OPD practical

One-sentence takeaway:

Offline distillation = memorize the teacher's finished answers.
OPD = write first, then get graded line by line.
